# Clean Data

This step is for correct, impute or remove erroneous values.

In [1]:
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf
from olist_ml.config import OlistConfig
from sqlalchemy import create_engine

root_path = project_root=Path.cwd().parent.parent
cfg = OmegaConf.load("../../conf/config.yaml")

raw_data_path = root_path / cfg.paths.raw_data
olist_sqlite_path = raw_data_path / cfg.files.olist_sqlite
db_file = f"sqlite:///{olist_sqlite_path}"
engine = create_engine(db_file)
products = pd.read_sql("SELECT * FROM products;", engine)
order_items = pd.read_sql("SELECT * FROM order_items;", engine)

## 1. Product Category Name Missing Values

`product_category_name` is an important attribute because it allows filtering order items and analyzing demand at the product-category level.

In [2]:
products_imputed = products.copy()
products_imputed["product_category_name"] = products_imputed["product_category_name"].fillna("missing")
products.isna().sum(), products_imputed.isna().sum(), products_imputed[products_imputed["product_category_name"] == "missing"]["product_category_name"].head()

(product_id                      0
 product_category_name         610
 product_name_lenght           610
 product_description_lenght    610
 product_photos_qty            610
 product_weight_g                2
 product_length_cm               2
 product_height_cm               2
 product_width_cm                2
 dtype: int64,
 product_id                      0
 product_category_name           0
 product_name_lenght           610
 product_description_lenght    610
 product_photos_qty            610
 product_weight_g                2
 product_length_cm               2
 product_height_cm               2
 product_width_cm                2
 dtype: int64,
 105    missing
 128    missing
 145    missing
 154    missing
 197    missing
 Name: product_category_name, dtype: object)

Missing values in `product_category_name` are imputed using a new placeholder category label `missing`.

## 2. Order Items `price` and `freight_value` outlier values

In [3]:
order_items_no_outliers = order_items[
    (order_items["price"] <= 800) &
    (order_items["freight_value"] <= 80)
]
order_items[["price", "freight_value"]].describe(), order_items_no_outliers[["price", "freight_value"]].describe()

(               price  freight_value
 count  112650.000000  112650.000000
 mean      120.653739      19.990320
 std       183.633928      15.806405
 min         0.850000       0.000000
 25%        39.900000      13.080000
 50%        74.990000      16.260000
 75%       134.900000      21.150000
 max      6735.000000     409.680000,
                price  freight_value
 count  110239.000000  110239.000000
 mean      103.914659      18.670599
 std       104.332388      10.606085
 min         0.850000       0.000000
 25%        39.900000      12.950000
 50%        70.970000      16.140000
 75%       129.900000      20.640000
 max       800.000000      79.970000)

Outlier values for `price` and `freight_value` identified during the `verify data quality` step were removed from the `order_items` dataset. This ensures that extreme or erroneous values do not distort next steps.